In [ ]:
!pip install PyMC

In [80]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_regression, SelectKBest, SelectFromModel
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, ARDRegression, LassoLarsCV
import pymc as pm
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor 
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [55]:
df = pd.read_csv('imputed_iso.csv')

In [56]:
len(df)

2627

In [57]:
df.head(20)

,Dev Time (Days),Start Date,total_swag,impacted_area_(none),impacted_area_3rd Party,impacted_area_AMS,impacted_area_ASTRO Infrastructure,impacted_area_Accessories,impacted_area_Architecture,impacted_area_Audio,...,Engineering,One list,PDM,Parity/Evolution,Quality,Quality-ENG,Roadmap/PCN,SLT,Security,subtask_count
0,31.0,20200609.0,358.443,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,31.0,20200609.0,35.389,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,31.0,20200609.0,66.800,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,31.0,20200609.0,35.389,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,31.0,20200609.0,35.389,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,31.0,20200914.0,24.657,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,31.0,20210719.0,9.994,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,31.0,20210719.0,9.994,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,31.0,20210719.0,9.994,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,31.0,20230502.0,20.644,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Method 1: Mutual Information (Filter method)

In [58]:
X = df.drop(columns=['Dev Time (Days)'], axis=1)
y = df['Dev Time (Days)']



In [59]:
mi_scores = mutual_info_regression(X, y, random_state=42)

mi_series = pd.Series(mi_scores, index=X.columns, name="MI Scores")
mi_series = mi_series.sort_values(ascending=False)

print("\nMutual Information Regression Scores:")
print(mi_series)


Mutual Information Regression Scores:
Start Date                        2.021601
total_swag                        1.162246
fix_version_(no version)          0.108838
technology_(none)                 0.103088
impacted_area_(none)              0.087136
                                    ...   
fix_version_ASTRO_SR7.18.8 CSR    0.000000
Engineering                       0.000000
PDM                               0.000000
Quality                           0.000000
SLT                               0.000000
Name: MI Scores, Length: 176, dtype: float64


In [60]:
len(mi_series)

176

We select features with mi score > 0.01

In [61]:
good_features = mi_series[mi_series > 0.01].index


X_mutual_info = X[good_features]

print(X_mutual_info.columns)

listm = X_mutual_info.columns
len(listm)

Index(['Start Date', 'total_swag', 'fix_version_(no version)',
       'technology_(none)', 'impacted_area_(none)',
       'affects_version_(no version)', 'subtask_count', 'One list',
       'technology_APX', 'technology_Aloha', 'fix_version_ASTRO_SR2022.2',
       'fix_version_ASTRO_SR2023.1MR2', 'fix_version_ASTRO_SR7.18.5',
       'fix_version_ASTRO_SR7.18.8', 'affects_version_ASTRO_SR2023.1MR2',
       'fix_version_ASTRO_SR2023.1', 'fix_version_ASTRO_SR7.18 MOL',
       'fix_version_ASTRO_SR2019.2', 'impacted_area_Box Test',
       'impacted_area_DSP / Audio', 'fix_version_ASTRO_SR2022.2MR1',
       'Cyber RR', 'affects_version_ASTRO_SR2023.2',
       'impacted_area_Orderability', 'impacted_area_SWBB Connectivity',
       'impacted_area_AMS', 'affects_version_ASTRO_SR2021.2',
       'fix_version_No_SW_Release', 'impacted_area_IDM', 'impacted_area_IMW',
       'impacted_area_CPS/RM', 'affects_version_ASTRO_SR2025.1',
       'impacted_area_BDP', 'fix_version_ASTRO_SR2020.4',
       'i

45

# Method 2: Lasso Regression (Embedded Method)

First, scale the data

In [62]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X, y)

In [63]:
lasso_cv = LassoCV(cv=5, random_state=42)

In [64]:
selector = SelectFromModel(estimator=lasso_cv, threshold=1e-5)

In [65]:
selector.fit(X_scaled, y)

C:\Users\KRG836\AppData\Local\anaconda3\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:681: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 22231.209099158645, tolerance: 13582.814257877202
  model = cd_fast.enet_coordinate_descent_gram(


SelectFromModel(estimator=LassoCV(cv=5, random_state=42), threshold=1e-05)

In [66]:
selected_cols = X.columns[selector.get_support()]
X_lasso = X[selected_cols]
print(X_lasso.columns)

list2 = X_lasso.columns
len(list2)

Index(['Start Date', 'impacted_area_(none)', 'One list', 'subtask_count'], dtype='object')


4

# Method 3: Automatic Relevancy Determination (Embedded Method - Bayesian)

In [67]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit ARD Regression model
ard = ARDRegression(compute_score=True)
ard.fit(X_train, y_train)

# Extract feature weights and select relevent features
feature_weights = pd.Series(ard.coef_, index=X.columns)
relevant_features = feature_weights[np.abs(feature_weights) > 0.01]

print(f"Total initial features: {len(X.columns)}")
print(f"Features kept by ARD: {len(relevant_features)}")
print(f"Current n/p ratio: {len(X_train) / len(relevant_features):.2f}")


Total initial features: 176
Features kept by ARD: 70
Current n/p ratio: 30.01


# Method 4: Submodular Optimization (Wrapper)

In [68]:
import numpy as np
import pandas as pd
from sklearn.metrics import mutual_info_score
from sklearn.feature_selection import mutual_info_regression

import numpy as np
import pandas as pd
from sklearn.metrics import mutual_info_score
from sklearn.feature_selection import mutual_info_regression

def submodular_greedy_selection(X_data, y_target, k):
    all_features = list(X_data.columns)
    selected_indices = []
    remaining_indices = list(range(len(all_features)))
    
    k = min(k, len(all_features))
    print(f"--- Starting Submodular Selection (Budget k={k}) ---")

    for i in range(k):
        best_gain = -np.inf
        best_feature_idx = -1

        for idx in remaining_indices:
            # RELEVANCE: Always use regression MI for continuous target y
            current_feat_vals = X_data.iloc[:, [idx]].values
            relevance = mutual_info_regression(current_feat_vals, y_target, random_state=42)[0]
            
            # REDUNDANCY: Logic branch based on feature type
            redundancy = 0
            if selected_indices:
                red_scores = []
                for s in selected_indices:
                    feat_a = X_data.iloc[:, idx]
                    feat_b = X_data.iloc[:, s]
                    
                    # If BOTH are binary/categorical, use mutual_info_score (Discrete)
                    if len(np.unique(feat_a)) < 10 and len(np.unique(feat_b)) < 10:
                        score = mutual_info_score(feat_a, feat_b)
                    else:
                        # If either is continuous, use regression-based MI (Continuous)
                        # We reshape to (n, 1) for the regression function
                        score = mutual_info_regression(feat_a.values.reshape(-1, 1), feat_b, random_state=42)[0]
                    
                    red_scores.append(score)
                redundancy = np.mean(red_scores)
            
          
            gain = relevance - redundancy

            if gain > best_gain:
                best_gain = gain
                best_feature_idx = idx

        if best_feature_idx != -1:
            # EARLY STOPPING: If gain is negative, adding the feature makes the set worse
            if best_gain <= 0:
                print(f"--- Stopping early at Step {i+1}: Gain became non-positive ---")
                break
                
            selected_indices.append(best_feature_idx)
            remaining_indices.remove(best_feature_idx)
            print(f"Step {i+1}: Added '{all_features[best_feature_idx]}' | Gain: {best_gain:.4f}")
        else:
            break

    final_selection = [all_features[i] for i in selected_indices]
    print(f"\nFinal Feature Set ({len(final_selection)}): {final_selection}")
    return final_selection


final_features = submodular_greedy_selection(X, y, k=18) # Set k=18 because gain turned negative at k=19


--- Starting Submodular Selection (Budget k=18) ---
Step 1: Added 'Start Date' | Gain: 2.0343
Step 2: Added 'total_swag' | Gain: 0.3289
Step 3: Added 'fix_version_ASTRO_SR2022.2' | Gain: 0.0121
Step 4: Added 'fix_version_ASTRO_SR2024.2MR' | Gain: 0.0074
Step 5: Added 'fix_version_(no version)' | Gain: 0.0225
Step 6: Added 'impacted_area_(none)' | Gain: 0.0274
Step 7: Added 'One list' | Gain: 0.0113
Step 8: Added 'fix_version_ASTRO_SR2023.1' | Gain: 0.0079
Step 9: Added 'technology_Aloha' | Gain: 0.0100
Step 10: Added 'technology_APX Next' | Gain: 0.0096
Step 11: Added 'technology_APX' | Gain: 0.0098
Step 12: Added 'fix_version_ASTRO_SR2019.2' | Gain: 0.0072
Step 13: Added 'technology_(none)' | Gain: 0.0083
Step 14: Added 'fix_version_ASTRO_SR7.18.8' | Gain: 0.0077
Step 15: Added 'fix_version_ASTRO_SR2020.4' | Gain: 0.0057
Step 16: Added 'fix_version_ASTRO_SR7.18.5' | Gain: 0.0036
Step 17: Added 'affects_version_ASTRO_SR2021.3' | Gain: 0.0025
Step 18: Added 'affects_version_ASTRO_SR2022

In [ ]:
import pandas as pd

n_bootstraps = 1000
n_size = len(df)
bootstrap_samples = [] 

for i in range(n_bootstraps):
    sample = df.sample(n=n_size, replace=True)
    
    bootstrap_samples.append(sample)

print("--- First Bootstrap Sample ---")
print(bootstrap_samples[0])

In [ ]:
X = X_mutual_info
y = np.log1p(df['Dev Time (Days)'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = ExtraTreesRegressor(n_estimators =500, random_state =42)
model.fit(X_train, y_train)

print("Model trained successfully!")
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print(f"Mean Absolute Error (MAE): {mae:.2f}")


r2 = r2_score(y_test, predictions)
print(f"R-squared Score: {r2:.2f}")

In [ ]:
X = X_lasso
y = np.log1p(df['Dev Time (Days)'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = ExtraTreesRegressor(n_estimators =500, random_state =42)
model.fit(X_train, y_train)

print("Model trained successfully!")
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
print(f"Mean Absolute Error (MAE): {mae:.2f}")


r2 = r2_score(y_test, predictions)
print(f"R-squared Score: {r2:.2f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import ExtraTreesRegressor

target = 'Dev Time (Days)'
y = np.log1p(df[target])

model = ExtraTreesRegressor(n_estimators=500, random_state=42, n_jobs=-1)

cv = KFold(n_splits=10, shuffle=True, random_state=42)

print("Running Cross-Validation for Mutual Info features...")
mi_scores = cross_val_score(model, X_mutual_info, y, cv=cv, scoring='r2')

print(f"\nMutual Info - R-squared scores for each fold: {mi_scores}")
print(f"Mutual Info - Mean R-squared: {mi_scores.mean():.4f}")
print(f"Mutual Info - Std Deviation: {mi_scores.std():.4f}")


print("\nRunning Cross-Validation for Lasso features...")
lasso_scores = cross_val_score(model, X_lasso, y, cv=cv, scoring='r2')

print(f"\nLasso - R-squared scores for each fold: {lasso_scores}")
print(f"Lasso - Mean R-squared: {lasso_scores.mean():.4f}")
print(f"Lasso - Std Deviation: {lasso_scores.std():.4f}")

# Deep Knock-Off

In [69]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np

class KnockoffVAE(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super(KnockoffVAE, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim * 2) # Mean and variance
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim),
            nn.Sigmoid() # Keeps values between 0-1 (good for scaled data)
        )

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = torch.chunk(h, 2, dim=1)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

def generate_deep_knockoffs(X_df, epochs=100, latent_dim=10):
    X_min = X_df.min()
    X_max = X_df.max()
    # Handle columns with zero variance
    X_range = (X_max - X_min).replace(0, 1)
    X_scaled = (X_df - X_min) / X_range
    
    X_tensor = torch.FloatTensor(X_scaled.values)
    input_dim = X_tensor.shape[1]
    
    #Setup model
    model = KnockoffVAE(input_dim, latent_dim)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    #Training Loop
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        recon_x, mu, logvar = model(X_tensor)
        
        #Loss = Reconstruction Error + KL Divergence
        recon_loss = nn.functional.mse_loss(recon_x, X_tensor, reduction='sum')
        kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        loss = recon_loss + kld_loss
        
        loss.backward()
        optimizer.step()

    #Generate Knockoffs
    model.eval()
    with torch.no_grad():
        # Re-encode and sample to create the knockoff twins
        X_tilde_scaled, _, _ = model(X_tensor)
        X_tilde_np = X_tilde_scaled.numpy()
        
    #Reverse Scaling
    X_tilde = (X_tilde_np * X_range.values) + X_min.values
    
    #Clean up Binary columns (Clip to 0 or 1 for one-hot encoded fields)
    for i, col in enumerate(X_df.columns):
        if len(X_df[col].unique()) <= 2: 
            X_tilde[:, i] = np.where(X_tilde[:, i] > 0.5, 1, 0)

    return pd.DataFrame(X_tilde, columns=[f"knockoff_{c}" for c in X_df.columns])

X_knockoff = generate_deep_knockoffs(X)

In [70]:
X_combined = pd.concat([X,X_knockoff], axis=1)

In [72]:
mi_scores = mutual_info_regression(X_combined, y, random_state=42)

n_features = X.shape[1]
Z = mi_scores[:n_features] # real mi scores
Z_tilde = mi_scores[n_features:] # Knockoff mi scores

W = Z - Z_tilde # difference in information

final_mask = (Z > 0.01) & (W > 0)
valid_indices = np.where(final_mask)[0]

mi_good_features_knockoff = X.columns[valid_indices]
X_mutual_info_final = X[mi_good_features_knockoff]

print(f"Original MI selection count: {len(mi_series[mi_series > 0.01])}")
print(f"Knockoff-validated MI count: {len(mi_good_features_knockoff)}")
print("\nValidated Features:")
print(mi_good_features_knockoff.tolist())

Original MI selection count: 45
Knockoff-validated MI count: 41

Validated Features:
['Start Date', 'total_swag', 'impacted_area_(none)', 'impacted_area_Box Test', 'impacted_area_CPS/RM', 'impacted_area_DCE', 'impacted_area_IDM', 'impacted_area_Radio Central', 'impacted_area_SWBB Connectivity', 'impacted_area_Service Central', 'impacted_area_UI / UX', 'technology_(none)', 'technology_APX', 'technology_APX Next', 'technology_Aloha', 'technology_Common SW Blocks & Components', 'technology_Mahalo', 'affects_version_(no version)', 'affects_version_ASTRO_SR2021.1', 'affects_version_ASTRO_SR2021.3', 'affects_version_ASTRO_SR2022.1', 'affects_version_ASTRO_SR2022.2', 'affects_version_ASTRO_SR2024.2CSR', 'affects_version_No_SW_Release', 'fix_version_(no version)', 'fix_version_ASTRO_SR2019.1', 'fix_version_ASTRO_SR2019.2', 'fix_version_ASTRO_SR2020.4', 'fix_version_ASTRO_SR2022.1MR1', 'fix_version_ASTRO_SR2022.2', 'fix_version_ASTRO_SR2023.1', 'fix_version_ASTRO_SR2023.3MR1', 'fix_version_ASTR

In [74]:
selected_features_combined = submodular_greedy_selection(X_combined, y, k=36)


selection_results = {feat: i for i, feat in enumerate(selected_features_combined)}

real_winners = []
knockoff_interlopers = []
final_selection = []

original_features = X.columns.tolist()

print("\n--- Submodular Knockoff Comparison ---")

for feat in original_features:
    knock_feat = f"knockoff_{feat}"

    rank_real = selection_results.get(feat, float('inf'))
    rank_knockoff = selection_results.get(knock_feat, float('inf'))
    

    if rank_real < rank_knockoff:
        real_winners.append(feat)
        final_selection.append(feat)
    elif rank_knockoff < rank_real:
        knockoff_interlopers.append(feat)

print(f"Total original features tested: {len(original_features)}")
print(f"Features that beat their knockoff: {len(real_winners)}")
print(f"Features beaten by their knockoff (Noise): {len(knockoff_interlopers)}")

print("\nFinal Validated Feature Set:")
print(final_selection)

if knockoff_interlopers:
    print("\nRejected (Noise) Features:")
    print(knockoff_interlopers)

--- Starting Submodular Selection (Budget k=36) ---
Step 1: Added 'Start Date' | Gain: 2.0343
Step 2: Added 'total_swag' | Gain: 0.3289
Step 3: Added 'knockoff_Start Date' | Gain: 0.0339
Step 4: Added 'fix_version_ASTRO_SR2022.2' | Gain: 0.0196
Step 5: Added 'fix_version_(no version)' | Gain: 0.0242
Step 6: Added 'impacted_area_(none)' | Gain: 0.0265
Step 7: Added 'One list' | Gain: 0.0110
Step 8: Added 'fix_version_ASTRO_SR2024.2MR' | Gain: 0.0100
Step 9: Added 'technology_Aloha' | Gain: 0.0107
Step 10: Added 'technology_APX Next' | Gain: 0.0095
Step 11: Added 'fix_version_ASTRO_SR2023.1' | Gain: 0.0101
Step 12: Added 'technology_(none)' | Gain: 0.0149
Step 13: Added 'fix_version_ASTRO_SR2019.2' | Gain: 0.0083
Step 14: Added 'fix_version_ASTRO_SR7.18.8' | Gain: 0.0087
Step 15: Added 'technology_APX' | Gain: 0.0073
Step 16: Added 'fix_version_ASTRO_SR2020.4' | Gain: 0.0061
Step 17: Added 'knockoff_total_swag' | Gain: 0.0067
Step 18: Added 'fix_version_ASTRO_SR7.18.5' | Gain: 0.0042
Ste

In [84]:
X_combined_scaled = scaler.fit_transform(X_combined)

lasso_cv = LassoCV(
    n_alphas=200,
    cv=5, 
    random_state=42, 
    max_iter=50000,    
    tol=0.01         
)

lasso_cv.fit(X_combined_scaled,y)

n_features = X.shape[1]
abs_coefs = np.abs(lasso_cv.coef_)

z_real = abs_coefs[:n_features]
z_knockoff = abs_coefs[:n_features]

w_stats = z_real - z_knockoff
real_winners = X.columns[w_stats > 0].tolist()

beaten_by_noise = X.columns[z_knockoff > z_real].tolist()

print("--- Lasso Knockoff Validation ---")
print(f"Features originally in X: {n_features}")
print(f"Features that beat their twin: {len(real_winners)}")
print(f"Features beaten by noise: {len(beaten_by_noise)}")

print("\nFinal Validated Lasso Set:")
print(real_winners)

if beaten_by_noise:
    print("\nRejected Features (Interlopers):")
    print(beaten_by_noise)

--- Lasso Knockoff Validation ---
Features originally in X: 176
Features that beat their twin: 0
Features beaten by noise: 0

Final Validated Lasso Set:
[]


LASSO is not a good model for feature selection for this dataset

In [87]:
from sklearn.linear_model import ARDRegression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_comb_scaled = scaler.fit_transform(X_combined)

# Fit ARD
print("--- Starting ARD Knockoff Tournament ---")
ard_model = ARDRegression(compute_score=True)
ard_model.fit(X_comb_scaled, y)


n_features = X.shape[1]
weights = np.abs(ard_model.coef_)

z_real = weights[:n_features]           # Importance of real features
z_knockoff = weights[n_features:]      # Importance of synthetic twins

w_stats = z_real - z_knockoff
valid_mask = (w_stats > 0) & (z_real > 0.01)
real_winners_ard = X.columns[valid_mask].tolist()


print(f"Features originally in X: {n_features}")
print(f"ARD Survivors (Beat their knockoff): {len(real_winners_ard)}")

if len(real_winners_ard) > 0:
    print("\nValidated ARD Features:")
    print(real_winners_ard)
else:
    print("\nNo features survived the ARD competition.")

--- Starting ARD Knockoff Tournament ---
Features originally in X: 176
ARD Survivors (Beat their knockoff): 70

Validated ARD Features:
['Start Date', 'total_swag', 'impacted_area_(none)', 'impacted_area_3rd Party', 'impacted_area_ASTRO Infrastructure', 'impacted_area_Accessories', 'impacted_area_Auto Test', 'impacted_area_Blackbox Cloud', 'impacted_area_Box Test', 'impacted_area_CC Admin', 'impacted_area_CIE', 'impacted_area_CM', 'impacted_area_CMSO', 'impacted_area_CPS/RM', 'impacted_area_EE - Digital', 'impacted_area_EE - RF', 'impacted_area_Ergo', 'impacted_area_FutureCom', 'impacted_area_IDM', 'impacted_area_IT', 'impacted_area_IoT Hub', 'impacted_area_Kodiak', 'impacted_area_Low Level', 'impacted_area_Orderability', 'impacted_area_RC Insights', 'impacted_area_Regulatory', 'impacted_area_SPG', 'impacted_area_SWBB Android Core / Common', 'impacted_area_SWBB Connectivity', 'impacted_area_SWBB Low Level', 'impacted_area_Service Central', 'impacted_area_SmartConnect Gateway', 'technol

In [89]:
mutual = pd.concat([y,X[mi_good_features_knockoff]],axis=1)
ARD = pd.concat([y,X[real_winners_ard]],axis=1)
submodular = pd.concat([y,X[final_selection]],axis=1)

In [90]:
mutual.to_csv('iso_mutual_info.csv',index=False)
ARD.to_csv('iso_ard.csv',index=False)
submodular.to_csv('iso_submod.csv',index=False)